# Chapter 3 - Gaussian Filters: KF / EKF / UKF / IF

Three learning tiers:
1. Tier 1 - Toy 1D Kalman Filter: see Gaussian conditioning directly
2. Tier 2 - EKF with Jacobian finite-difference verification
3. Tier 3 - UKF vs EKF on figure-8, plus Information Filter

The Kalman gain as a trust ratio:

    K = Sigma_prior @ C.T @ inv(C @ Sigma_prior @ C.T + Q)

When Q is small, the measurement pulls hard.
When Sigma_prior is small, the prior is rigid and measurements have less effect.

In [ ]:
import sys
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_filters')
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_experiments')
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse

from pluto_filters.kalman_filters.ekf import EKF, LANDMARKS, normalize_angle
from pluto_filters.kalman_filters.ukf import UKF
from pluto_experiments.filter_showdown.benchmark import (
    generate_figure8_trajectory, generate_measurements, rmse, nees
)

def draw_cov_ellipse(ax, mu, sigma, n_std=2, **kw):
    vals, vecs = np.linalg.eigh(sigma[:2, :2])
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w = 2 * n_std * np.sqrt(max(vals[0], 1e-10))
    h = 2 * n_std * np.sqrt(max(vals[1], 1e-10))
    ax.add_patch(Ellipse(xy=mu[:2], width=w, height=h, angle=angle, **kw))

print("Setup complete")

## Tier 1 - 1D Kalman Filter: Gaussian Conditioning

In [ ]:
np.random.seed(42)
N_STEPS  = 40
TRUE_VEL = 0.2
DT_1D    = 0.5
R_var    = 0.5    # measurement noise variance
Q_var    = 0.01   # process noise variance

true_pos = np.cumsum(np.ones(N_STEPS) * TRUE_VEL * DT_1D)
measures = true_pos + np.random.normal(0, np.sqrt(R_var), N_STEPS)

mu, sigma = 0.0, 10.0
mu_hist, sigma_hist, K_hist = [mu], [sigma], []

for z in measures:
    mu    = mu + TRUE_VEL * DT_1D
    sigma = sigma + Q_var
    K     = sigma / (sigma + R_var)
    mu    = mu + K * (z - mu)
    sigma = (1 - K) * sigma
    K_hist.append(K)
    mu_hist.append(mu)
    sigma_hist.append(sigma)

mu_arr  = np.array(mu_hist[1:])
sig_arr = np.array(sigma_hist[1:])
t_arr   = np.arange(1, N_STEPS + 1) * DT_1D

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Tier 1 - 1D Kalman Filter', fontsize=13)

ax = axes[0]
ax.fill_between(t_arr, mu_arr - 2*sig_arr, mu_arr + 2*sig_arr,
                alpha=0.25, color='steelblue', label='+/- 2sigma')
ax.plot(t_arr, true_pos,  'k-', lw=2, label='Ground truth')
ax.plot(t_arr, measures,  'g.', ms=6, alpha=0.7, label='Measurements')
ax.plot(t_arr, mu_arr,    'b-', lw=2, label='KF estimate')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Position [m]')
ax.set_title('KF tracks truth within +/-2sigma'); ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_arr, K_hist, 'purple', lw=2)
ax.set_xlabel('Time [s]'); ax.set_ylabel('Kalman Gain K')
ax.set_title('K converges to steady state\nK near 1 = trust measurement\nK near 0 = trust prior')

ax = axes[2]
ax.plot(t_arr, sig_arr, 'teal', lw=2)
ax.set_xlabel('Time [s]'); ax.set_ylabel('Uncertainty sigma')
ax.set_title('sigma converges from wide prior')

plt.tight_layout()
plt.savefig('ch03_tier1_kf.png', dpi=120)
plt.show()
print(f"Initial sigma={10.0:.1f} -> Final sigma={sig_arr[-1]:.4f}")
print(f"Final Kalman gain: {K_hist[-1]:.4f}")

## Tier 2 - EKF: Jacobian Finite-Difference Verification

Before using an analytic Jacobian in an EKF, verify it against finite differences.
If the max absolute error between analytic and numerical Jacobian exceeds 1e-5,
the linearization is likely wrong.

In [ ]:
def measurement_model(pose, lx, ly):
    dx  = lx - pose[0]; dy = ly - pose[1]
    r   = np.sqrt(dx**2 + dy**2)
    phi = normalize_angle(np.arctan2(dy, dx) - pose[2])
    return np.array([r, phi])

def jacobian_analytic(pose, lx, ly):
    dx  = lx - pose[0]; dy = ly - pose[1]
    r2  = dx**2 + dy**2; r = np.sqrt(r2)
    return np.array([
        [-dx/r,   -dy/r,  0],
        [ dy/r2, -dx/r2, -1]
    ])

def jacobian_finitediff(pose, lx, ly, eps=1e-5):
    H = np.zeros((2, 3))
    for i in range(3):
        dp = pose.copy(); dp[i] += eps
        dm = pose.copy(); dm[i] -= eps
        H[:, i] = (measurement_model(dp, lx, ly) - measurement_model(dm, lx, ly)) / (2*eps)
    return H

print("Jacobian check: analytic vs finite-difference\n")
test_poses = [
    np.array([1.0,  0.5,  0.3]),
    np.array([2.0, -1.0,  1.2]),
    np.array([-1.0, 2.0, -0.7]),
]

all_pass = True
for pose in test_poses:
    for lm_id, (lx, ly) in LANDMARKS.items():
        H_a  = jacobian_analytic(pose, lx, ly)
        H_fd = jacobian_finitediff(pose, lx, ly)
        err  = np.abs(H_a - H_fd).max()
        ok   = err < 1e-5
        if not ok: all_pass = False
        print(f"{'PASS' if ok else 'FAIL'}  pose=({pose[0]:.1f},{pose[1]:.1f},{pose[2]:.1f})"
              f"  LM{lm_id}  max|H_analytic - H_fd| = {err:.2e}")
print(f"\nAll checks passed: {all_pass}")

## Tier 3 - EKF vs UKF on figure-8 trajectory

In [ ]:
np.random.seed(42)
true_poses, dt = generate_figure8_trajectory(300, 0.1)
measurements   = generate_measurements(true_poses, landmark_id=0)

def run_filter(filter_cls, v=0.3, omega=0.2):
    mu0    = np.array([0.3, 0.2, 0.15])
    sigma0 = np.diag([1.0, 1.0, 0.2])
    f      = filter_cls(mu0, sigma0)
    ests, covs = [], []
    for i, (r, phi) in enumerate(measurements):
        if i > 0:
            f.predict(v, omega, dt)
        f.update(0, r, phi)
        ests.append(f.mu.copy())
        covs.append(f.sigma.copy())
    return np.array(ests), covs

ekf_ests, ekf_covs = run_filter(EKF)
ukf_ests, ukf_covs = run_filter(UKF)

ekf_rmse = rmse(ekf_ests, true_poses)
ukf_rmse = rmse(ukf_ests, true_poses)
ekf_nees = nees(ekf_ests, ekf_covs, true_poses)
ukf_nees = nees(ukf_ests, ukf_covs, true_poses)

print(f"EKF: RMSE={ekf_rmse:.4f}m  NEES={ekf_nees:.3f}")
print(f"UKF: RMSE={ukf_rmse:.4f}m  NEES={ukf_nees:.3f}")
print(f"NEES should be close to 3 (state dimension) for a consistent filter.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('EKF vs UKF - figure-8 with covariance ellipses', fontsize=13)

for ax, ests, covs, name, color in [
    (axes[0], ekf_ests, ekf_covs, 'EKF', 'steelblue'),
    (axes[1], ukf_ests, ukf_covs, 'UKF', 'coral'),
]:
    ax.plot(true_poses[:,0], true_poses[:,1], 'k-', lw=2, label='Ground Truth', zorder=5)
    ax.plot(ests[:,0], ests[:,1], '--', color=color, lw=1.5, label=f'{name} estimate')
    for i in range(0, len(ests), 20):
        try:
            draw_cov_ellipse(ax, ests[i], covs[i], n_std=2,
                             fill=False, edgecolor=color, alpha=0.4, lw=1)
        except Exception:
            pass
    for lm_id, (lx, ly) in LANDMARKS.items():
        ax.plot(lx, ly, 'k^', ms=9)
        ax.annotate(f'LM{lm_id}', (lx+0.1, ly+0.1), fontsize=7)
    rmse_val = rmse(ests, true_poses)
    nees_val = nees(ests, covs, true_poses)
    ax.set_title(f'{name}  RMSE={rmse_val:.4f}m  NEES={nees_val:.2f}')
    ax.legend(fontsize=8); ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout()
plt.savefig('ch03_gaussian_filters.png', dpi=120)
plt.show()

## Tier 3 continued - Information Filter

In [ ]:
# Information Filter in 1D: Omega = Sigma^{-1}, xi = Omega @ mu
# Each measurement adds information (increases Omega)
# Sensor fusion: Omega_fused = Omega_1 + Omega_2 (just addition)

Omega   = 1e-4   # near-zero = nearly uniform prior
xi      = 0.0    # information vector
IF_mus, IF_omegas = [], []

for z in measures:
    # Predict: process noise reduces information
    Sigma = 1.0 / (Omega + 1e-12) + Q_var
    Omega = 1.0 / Sigma
    xi    = Omega * (xi / (Omega + 1e-12))

    # Update: measurement adds information
    Omega_z = 1.0 / R_var
    Omega   = Omega + Omega_z
    xi      = xi + Omega_z * z

    IF_mus.append(xi / Omega)
    IF_omegas.append(Omega)

IF_mus = np.array(IF_mus)
IF_sig = 1.0 / np.sqrt(np.array(IF_omegas))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Tier 3 - Information Filter: Omega grows with measurements', fontsize=13)

ax = axes[0]
ax.fill_between(t_arr, IF_mus - 2*IF_sig, IF_mus + 2*IF_sig,
                alpha=0.2, color='teal', label='+/- 2sigma')
ax.plot(t_arr, true_pos, 'k-', lw=2, label='Ground truth')
ax.plot(t_arr, IF_mus,   'teal', lw=2, label='IF estimate')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Position [m]')
ax.set_title('IF estimate vs truth'); ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_arr, IF_omegas, 'teal', lw=2)
ax.set_xlabel('Time [s]'); ax.set_ylabel('Information Omega = Sigma^{-1}')
ax.set_title('Omega grows with each measurement\nFusion of two independent sensors: Omega_1 + Omega_2')

plt.tight_layout()
plt.savefig('ch03_tier3_if.png', dpi=120)
plt.show()

## Compare, Break, Measure

In [ ]:
print("=== Compare: EKF vs UKF ===")
print(f"{'Filter':>6}  {'RMSE [m]':>10}  {'NEES':>8}  {'Consistent (NEES~3)?':>22}")
print("-" * 52)
for name, r_, n_ in [('EKF', ekf_rmse, ekf_nees), ('UKF', ukf_rmse, ukf_nees)]:
    ok = 'YES' if abs(n_ - 3.0) < 3.0 else 'NO - overconfident' if n_ > 3 else 'NO - underconfident'
    print(f"{name:>6}  {r_:>10.4f}  {n_:>8.3f}  {ok:>22}")
print()
print("RMSE measures accuracy. NEES measures whether the reported uncertainty is honest.")
print("A filter with low RMSE but high NEES is accurate but overconfident.")

print("\n=== Break: increase initial uncertainty ===")
for sigma_init in [1.0, 5.0, 20.0]:
    ekf_b = EKF(np.array([0.3, 0.2, 0.15]), np.eye(3) * sigma_init)
    ests_b = []
    for i, (r, phi) in enumerate(measurements):
        if i > 0: ekf_b.predict(0.3, 0.2, 0.1)
        ekf_b.update(0, r, phi)
        ests_b.append(ekf_b.mu.copy())
    print(f"  sigma_init={sigma_init:.1f}: EKF RMSE={rmse(np.array(ests_b), true_poses):.4f}m")

print("\n=== Measure: NEES interpretation ===")
print("  NEES << n_state: filter is underconfident (uncertainty too large)")
print("  NEES ~= n_state: filter is consistent")
print("  NEES >> n_state: filter is overconfident (uncertainty too small)")
print(f"  State dimension n=3, EKF NEES={ekf_nees:.2f}, UKF NEES={ukf_nees:.2f}")